# L'isola del giorno prima — isola.json

Pipeline:
1. Legge `isola.pdf`
2. GLiNER estrae oggetti, personaggi e concetti (etichette estese per coprire i 6 topic)
3. Deduplica per oggetto unico
4. Filtra il rumore (soglia GLiNER più alta + pre-filtro coseno grezzo)
5. sentence-transformers categorizza ogni oggetto sui 6 topic
6. Esporta `isola.json`

**6 topic indipendenti della mostra** (non sequenziali — rif. Eco, Lector in Fabula):

- **lista_inventario** — wunderkammer: uccelli, pesci, piante, conchiglie, coralli, orologi, armi, stelle
- **invenzione_scoperta** — polvere di simpatia, cannocchiale, carte geografiche, longitudine, Punto Fijo
- **artificiale_naturale** — teatro della memoria, libro, statua, sostanza, geroglifici, mente estesa
- **misura_infinito** — pendolo, astrolabio, musica, cosmologia, eternità, macchie lunari, calcolo
- **potere_sapere** — corte, guerra, Gesuiti, dissimulazione, Mazzarino, Richelieu, ragion di stato
- **autore_doppio** — Ferrante, sosia, sogno, pensieri impercettibili, Lilia, Tweede Daphne

Termini **in giallo** (generici, contestuali): uccelli, pesci, piante, mostri, conchiglie, coralli,
venti, navi, armi, stelle, sostanza — presenti in più topic, disambiguati dal contesto estratto.

In [123]:
# !pip install gliner sentence-transformers pdfplumber scikit-learn tqdm

In [124]:
import json, re
import numpy as np
import pdfplumber
from pathlib import Path
from tqdm.auto import tqdm
from gliner import GLiNER
from sentence_transformers import SentenceTransformer

In [ ]:
PDF_PATH     = Path('../isola.pdf')
OUT_PATH     = Path('../app/static/isola.json')
GLINER_MODEL = 'urchade/gliner_multi-v2.1'
ST_MODEL     = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'

GLINER_BATCH = 32
THRESHOLD    = 0.28
MIN_CHARS    = 30
MAX_CHARS    = 500
MIN_FREQ     = 1
MAX_OBJECTS  = 12000

# GLiNER: tipi di entità fisiche e simboliche del romanzo
GLINER_LABELS = [
    'strumento scientifico o di misura',        # astrolabio, sestante, pendolo, cronometro
    'strumento ottico',                          # cannocchiale, lente, specchio
    'imbarcazione o parte di nave',              # scialuppa, vela, ancora, timone
    'oggetto religioso o sacro',                 # crocifisso, rosario, bibbia, reliquia
    'libro, lettera o documento scritto',        # lettera, libro, mappa, manoscritto
    'animale selvatico o esotico',               # colomba, uccello del paradiso, capre
    'creatura marina o corallina',               # corallo, conchiglia, tartaruga, pesce
    'pianta, fiore o frutto esotico',            # rosa, orchidea, frutto tropicale
    'arma o strumento militare',                 # spada, pistola, cannone, pugnale
    'oggetto fisico concreto',                   # corda, barile, campana, maschera
    'fenomeno celeste o atmosferico',            # stella, luna, cometa, venti, meteore
    "oggetto personale o pegno d'amore",         # ritratto, anello, medaglia
    'personaggio storico o letterario',          # Mazzarino, Ferrante, Richelieu, Lilia
    'concetto filosofico o metafisico',          # sostanza, vuoto, eternità, mente estesa
    'luogo geografico, città o istituzione',     # Casale, isola di Vesavio, museo wormiano, corte
    'opera musicale, artistica o performativa',  # musica, pavana, quadri, statue, strumenti musicali
    'organizzazione, ordine o corpo militare',   # Gesuiti, Compagnia di Gesù, reggimento, esercito
]

# Etichette descrittive per il meta JSON (non usate per il calcolo ML).
# Il calcolo è guidato da PROTOTYPES in cella successiva.
CATEGORIE = {
    'lista_inventario':
        'wunderkammer, uccelli, pesci, piante, isole, mostri, orologi, conchiglie, coralli, '
        'pietre, stelle, venti, strumenti musicali, navi, strumenti nautici, '
        'armi, pallottole, pugnale, spada, pistola, cannone',
    'invenzione_scoperta':
        'polvere di simpatia, carte geografiche, geografia e idrografia, '
        'cannocchiale aristotelico, lente, specchio, latitudine, longitudine, pianeti, '
        'mondi sotterranei e plurali, inferno, cannocchiale galileo, rotte, '
        "instrumentum arcetricum, d'Igby, Punto Fijo",
    'artificiale_naturale':
        'scrittura, teatro della memoria, metafora, romanzo, lettera, libro, penne, inchiostri, '
        'compassi, globi, squadre, palagi, templi e tuguri, scudi, tamburi, quadri, pennelli, statue, '
        'sostanza, manoscritto, mappa, portolano, breviario, uccelli, colombe, pesci, piante, isole, '
        'mostri, orologi, conchiglie, coralli, pietre, luna, stelle, comete crinite barbate e codate, '
        'capre, travi, faci e saette, geroglifici, prati, uomini animati, meteore, venti, '
        'scherma, armi, archibugio, pallottole, pugnale, spada, pistola, cannone, '
        'campana, maschera, orchidea, frutto tropicale, rosa, peste, malattia, mente estesa',
    'misura_infinito':
        'musica, pavana, astrolabio, sestante, pendolo, cronometro, cannocchiale aristotelico, '
        'museo wormiano, lullo, ruote lulliane, clessidra, calcolo, stelle, geografia e idrografia, '
        'cosmologia, orologi oscillatori, orologio cattolico, mappamondo, rotte, isole Salomone, '
        'vuoto, limite, tempo, spazio, sostanza, estensione, universo, ore, minuti, millenni, '
        'atomi, eternità, macchie lunari, Galilei, Gassendi',
    'potere_sapere':
        "arte di prudenza, dissimulazione, ideale dell'onestuomo, conquista, assedio, Casale, "
        'Mazzarino, Richelieu, pitocchi, longitudine, ordigni, salotto, corte, re Sole, '
        'Pozzo di San Patrizio, Savoia, Guastalla, Gonzaga, Spinola, moschettieri, '
        'armi, pallottole, pugnale, spada, pistola, archibugio, politica, ragione di stato, '
        "Salazar, prudenza, ingegno, guerra, filosofia, scienza d'arme, Toiras, re cattolici, "
        'cardinale, missione, nuovo mondo, Punto Fijo, Colbert, potere, sapere, breviario, '
        'Compagnia di Gesù, Gesuiti, libertino, tradimento, spagnoli, Columbat, reggimento, esercito, '
        "Pompadour, psiche, Saint Savin, coup de la mouette, religione, popolo, re, "
        "madame de Rambouillet, d'Igby, Biscarat",
    'autore_doppio':
        'la signora, Lilia, paese dei romanzi, macchie lunari, Ferrante, fratello, sosia, '
        'pensieri impercettibili, personaggio, sogno, dottor Byrd, genio maligno, morte, mondo, '
        'isola di Vesavio, Historia, granseola, Giuda, Tweede Daphne, terra dei morti, '
        'Biscarat, pietra, coscienza, eroe segreto, Andropodo, Boride, Ordogno, Safar, Aspran',
}

## 1. Estrazione testo e segmentazione capitoli

In [126]:
CHAPTER_RE = re.compile(
    r'(?:^|\n)[ \t]*(\d{1,2})\.[ \t]+([A-ZÀÈÌÒÙL][^\n]{2,80})[ \t]*(?:\n|$)'
)

pages = []
with pdfplumber.open(PDF_PATH) as pdf:
    print(f'Pagine: {len(pdf.pages)}')
    for i, page in enumerate(tqdm(pdf.pages, desc='PDF')):
        t = page.extract_text(x_tolerance=2, y_tolerance=3)
        if t and t.strip():
            pages.append({'pdf_page': i + 1, 'text': t.strip()})

def segment_chapters(pages):
    chapters, current = [], {'numero': 0, 'titolo': '_intro', 'pagine': []}
    for p in pages:
        m = CHAPTER_RE.search(p['text'])
        if m:
            if current['pagine']: chapters.append(current)
            current = {'numero': int(m.group(1)), 'titolo': m.group(2).strip(), 'pagine': [p]}
        else:
            current['pagine'].append(p)
    if current['pagine']: chapters.append(current)
    return chapters

chapters = segment_chapters(pages)
seen, real_chapters = set(), []
for c in chapters:
    if c['numero'] > 0 and c['numero'] not in seen:
        seen.add(c['numero'])
        real_chapters.append(c)
real_chapters.sort(key=lambda c: c['numero'])
print(f'Capitoli: {len(real_chapters)}')

Pagine: 395


PDF: 100%|██████████| 395/395 [00:24<00:00, 15.92it/s]

Capitoli: 40


## 2. Segmentazione in frasi

In [127]:
def clean(t):
    t = re.sub(r'-\n', '', t)
    t = re.sub(r'\s+', ' ', t)
    return t.strip()

def split_sentences(text):
    parts = re.split(r'(?<=[a-z"\'\u2019])[.!?]+(?=[\s"\'\u201c]|$)', text)
    out = []
    for p in parts:
        p = p.strip()
        if MIN_CHARS <= len(p) <= MAX_CHARS:
            out.append(p)
        elif len(p) > MAX_CHARS:
            for sub in re.split(r'[;:]', p):
                sub = sub.strip()
                if MIN_CHARS <= len(sub) <= MAX_CHARS:
                    out.append(sub)
    return out

all_sentences = []
for c in tqdm(real_chapters, desc='Frasi'):
    text = clean(' '.join(p['text'] for p in c['pagine']))
    for s in split_sentences(text):
        all_sentences.append({
            'testo':           s,
            'capitolo':        c['numero'],
            'titolo_capitolo': c['titolo'],
        })

print(f'Frasi totali: {len(all_sentences):,}')

Frasi: 100%|██████████| 40/40 [00:00<00:00, 894.69it/s]

Frasi totali: 5,164


## 3. GLiNER — estrai oggetti concreti da ogni frase (deduplica come 04_macchina)

In [128]:
# Fix bug transformers: DebertaV2TokenizerFast chiama .endswith() su vocab_file=None
from transformers.convert_slow_tokenizer import convert_slow_tokenizer as _orig_convert
import transformers.convert_slow_tokenizer as _cst_module
import transformers.tokenization_utils_fast as _tuf

def _patched_convert(transformer_tokenizer, from_tiktoken=False):
    if getattr(transformer_tokenizer, 'vocab_file', None) is None:
        transformer_tokenizer.vocab_file = ''
    return _orig_convert(transformer_tokenizer, from_tiktoken=from_tiktoken)

_cst_module.convert_slow_tokenizer = _patched_convert
_tuf.convert_slow_tokenizer = _patched_convert

print(f'Caricamento GLiNER {GLINER_MODEL}...')
gliner = GLiNER.from_pretrained(GLINER_MODEL)
print('Pronto.')

Caricamento GLiNER urchade/gliner_multi-v2.1...


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 147686.76it/s]


Pronto.


In [ ]:
from collections import defaultdict, Counter

BAD_PREFIXES = (
    'il ', 'la ', 'lo ', 'i ', 'gli ', 'le ', "l'",
    'un ', 'una ', 'uno ',
    'nostra ', 'nostro ', 'nostri ', 'nostre ',
    'sua ', 'suo ', 'suoi ', 'sue ',
    'loro ', 'mio ', 'mia ', 'questo ', 'questa ',
    'quel ', 'quella ', 'quei ', 'quegli ',
)

# Personaggi chiave dei topic (Ferrante, Lilia, Mazzarino, ecc.) NON filtrati.
# 'mondo' NON filtrato — è termine di autore_doppio.
BAD_WORDS = {
    'uomo', 'uomini', 'donna', 'donne', 'gente', 'popolo',
    'ciurma', 'equipaggio', 'soldati', 'marinai', 'nemici', 'difensori',
    'selvaggi', 'indigeni', 'pirati', 'cavalieri', 'guerrieri',
    'roberto',
    'padre', 'figlio', 'madre', 'sorella', 'moglie', 'amante',
    'dio', 'signore', 'cristo', 'natura', 'animale', 'cosa', 'oggetto',
    'luogo', 'modo', 'parte', 'punto', 'fatto', 'caso', 'forma',
    'arma', 'amore', 'vita', 'corpo', 'anima',
    'ombra', 'luce', 'buio', 'notte', 'giorno', 'ora', 'momento',
    'acqua', 'terra', 'aria', 'fuoco',
}

def is_plausible_object(word):
    w = word.strip()
    if len(w) < 3 or len(w) > 40:
        return False
    if len(w.split()) > 4:
        return False
    wl = w.lower()
    if any(wl.startswith(p) for p in BAD_PREFIXES):
        return False
    if wl in BAD_WORDS:
        return False
    if not re.search(r'[a-zA-ZàèìòùÀÈÌÒÙ]', w):
        return False
    if re.match(r'^lettera\s+[a-z]$', wl):
        return False
    if w.isupper() and len(w) > 2:
        return False
    return True

all_sents = [s['testo'] for s in all_sentences]
mentions  = defaultdict(list)

for i in tqdm(range(0, len(all_sents), GLINER_BATCH), desc='GLiNER'):
    batch   = all_sents[i:i + GLINER_BATCH]
    results = gliner.batch_predict_entities(batch, GLINER_LABELS, threshold=THRESHOLD)
    for j, (sent, ents) in enumerate(zip(batch, results)):
        s = all_sentences[i + j]
        for ent in ents:
            word = ent['text'].strip()
            if not is_plausible_object(word):
                continue
            mentions[word.lower()].append((word, sent, s['capitolo'], s['titolo_capitolo']))

print(f'Entità candidate: {len(mentions)}')

resolved = []
for key, hits in mentions.items():
    if len(hits) < MIN_FREQ:
        continue
    canonical = Counter(h[0] for h in hits).most_common(1)[0][0]
    canonical = canonical[0].upper() + canonical[1:]
    best_hit  = max(hits, key=lambda h: len(h[1]))
    resolved.append({
        'oggetto':         canonical,
        'testo':           best_hit[1],
        'capitolo':        best_hit[2],
        'titolo_capitolo': best_hit[3],
        'n_menzioni':      len(hits),
    })

resolved.sort(key=lambda v: v['n_menzioni'], reverse=True)
resolved = resolved[:MAX_OBJECTS]
print(f'Oggetti selezionati: {len(resolved)}')
print('Top 30:', [v['oggetto'] for v in resolved[:30]])

## 4. sentence-transformers — classifica con prototipi concreti (MAX cosine)

In [ ]:
print(f'Caricamento {ST_MODEL}...')
st_model = SentenceTransformer(ST_MODEL)
print('Pronto.')

# Prototipi keyword — lista esatta per ogni topic.
# Termini gialli (generici) compaiono in più topic: il contesto della frase disambigua.
PROTOTYPES = {
    'lista_inventario': [
        'wunderkammer', 'uccelli', 'pesci', 'piante', 'isole', 'mostri',
        'orologi', 'conchiglie', 'coralli', 'pietre', 'stelle', 'venti',
        'strumenti musicali', 'navi', 'strumenti nautici',
        'armi', 'pallottole', 'pugnale', 'spada', 'pistola', 'cannone',
    ],

    'invenzione_scoperta': [
        'polvere di simpatia', 'carte geografiche', 'geografia e idrografia',
        'cannocchiale aristotelico', 'lente', 'specchio',
        'latitudine', 'longitudine', 'pianeti',
        'mondi sotterranei', 'mondi plurali', 'inferno',
        'cannocchiale galileo', 'rotte', 'instrumentum arcetricum',
        "d'Igby", 'Punto Fijo',
    ],

    'artificiale_naturale': [
        'scrittura', 'teatro della memoria', 'metafora', 'romanzo', 'lettera', 'libro',
        'penne', 'inchiostri', 'compassi', 'globi', 'squadre',
        'palagi', 'templi e tuguri', 'scudi', 'tamburi', 'quadri', 'pennelli', 'statue',
        'sostanza', 'manoscritto', 'mappa', 'portolano', 'breviario',
        'uccelli', 'colombe', 'pesci', 'piante', 'isole', 'mostri',
        'orologi', 'conchiglie', 'coralli', 'pietre',
        'luna', 'stelle', 'comete crinite', 'comete barbate', 'comete codate',
        'capre', 'travi', 'faci', 'saette', 'geroglifici', 'prati',
        'uomini animati', 'meteore', 'venti',
        'scherma', 'armi', 'archibugio', 'pallottole', 'pugnale', 'spada', 'pistola', 'cannone',
        'campana', 'maschera', 'orchidea', 'frutto tropicale', 'rosa',
        'peste', 'malattia', 'mente estesa',
    ],

    'misura_infinito': [
        'musica', 'pavana', 'astrolabio', 'sestante', 'pendolo', 'cronometro',
        'cannocchiale aristotelico', 'museo wormiano', 'lullo', 'ruote lulliane',
        'clessidra', 'calcolo', 'stelle', 'geografia e idrografia', 'cosmologia',
        'orologi oscillatori', 'orologio cattolico', 'mappamondo', 'rotte',
        'isole Salomone', 'vuoto', 'limite', 'tempo', 'spazio', 'sostanza',
        'estensione', 'universo', 'ore', 'minuti', 'millenni', 'atomi', 'eternità',
        'macchie lunari', 'Galilei', 'Gassendi',
    ],

    'potere_sapere': [
        'arte di prudenza', 'dissimulazione', "ideale dell'onestuomo",
        'conquista', 'assedio', 'Casale', 'Mazzarino', 'Richelieu',
        'pitocchi', 'longitudine', 'ordigni', 'salotto', 'corte', 're Sole',
        'Pozzo di San Patrizio', 'Savoia', 'Guastalla', 'Gonzaga', 'Spinola',
        'moschettieri', 'armi', 'pallottole', 'pugnale', 'spada', 'pistola', 'archibugio',
        'politica', 'ragione di stato', 'Salazar', 'prudenza', 'ingegno',
        'guerra', 'filosofia', "scienza d'arme", 'Toiras', 're cattolici',
        'cardinale', 'missione', 'nuovo mondo', 'Punto Fijo', 'Colbert',
        'potere', 'sapere', 'breviario', 'Compagnia di Gesù', 'Gesuiti',
        'libertino', 'tradimento', 'spagnoli', 'Columbat', 'reggimento', 'esercito',
        'Pompadour', 'psiche', 'Saint Savin', 'coup de la mouette',
        'religione', 'popolo', 're', 'madame de Rambouillet', "d'Igby", 'Biscarat',
    ],

    'autore_doppio': [
        'la signora', 'Lilia', 'paese dei romanzi', 'macchie lunari',
        'Ferrante', 'fratello', 'sosia', 'pensieri impercettibili',
        'personaggio', 'sogno', 'dottor Byrd', 'genio maligno',
        'morte', 'mondo', 'isola di Vesavio', 'Historia',
        'granseola', 'Giuda', 'Tweede Daphne', 'terra dei morti',
        'Biscarat', 'pietra', 'coscienza', 'eroe segreto',
        'Andropodo', 'Boride', 'Ordogno', 'Safar', 'Aspran',
    ],
}

cat_names = list(PROTOTYPES.keys())
TEMPERATURE   = 0.06
MIN_MAX_SCORE = 0.28
RAW_COS_MIN   = 0.15
ALPHA         = 0.60

print('Encoding prototipi...')
proto_embeddings = {}
for cat, protos in PROTOTYPES.items():
    embs = st_model.encode(protos, normalize_embeddings=True, show_progress_bar=False)
    proto_embeddings[cat] = embs

nomi = [v['oggetto'] for v in resolved]
ctx  = [f"{v['oggetto']}. {v['testo'][:120]}" for v in resolved]

print(f'Encoding {len(nomi)} oggetti (nome)...')
embs_nome = st_model.encode(nomi, normalize_embeddings=True, show_progress_bar=True)
print(f'Encoding {len(ctx)} oggetti (nome+contesto)...')
embs_ctx  = st_model.encode(ctx,  normalize_embeddings=True, show_progress_bar=True)

def max_sim_matrix(embs):
    out = np.zeros((len(embs), len(cat_names)))
    for j, cat in enumerate(cat_names):
        cos = embs @ proto_embeddings[cat].T
        out[:, j] = cos.max(axis=1)
    return out

raw_nome = max_sim_matrix(embs_nome)
raw_ctx  = max_sim_matrix(embs_ctx)
raw_sims = ALPHA * raw_nome + (1 - ALPHA) * raw_ctx

raw_max_per_obj = raw_sims.max(axis=1)
mask_raw = raw_max_per_obj >= RAW_COS_MIN
n_before_raw = len(resolved)
resolved = [v for v, keep in zip(resolved, mask_raw) if keep]
raw_sims = raw_sims[mask_raw]
print(f'Pre-filtro coseno grezzo (< {RAW_COS_MIN}): rimossi {n_before_raw - len(resolved)}, rimasti {len(resolved)}')

def softmax_temp(x, temp):
    x = x / temp
    x -= x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

scores = softmax_temp(raw_sims, TEMPERATURE)

max_scores = scores.max(axis=1)
mask_flat = max_scores >= MIN_MAX_SCORE
n_before_flat = len(resolved)
resolved = [v for v, keep in zip(resolved, mask_flat) if keep]
scores   = scores[mask_flat]
raw_sims = raw_sims[mask_flat]
print(f'Filtro piatto softmax (< {MIN_MAX_SCORE}): rimossi {n_before_flat - len(resolved)}, rimasti {len(resolved)}')

print('\nMedia per categoria:')
for cat, val in zip(cat_names, scores.mean(axis=0)):
    print(f'  {cat:<22} {val:.3f}  {"█" * int(val * 80)}')

from collections import Counter
dominant_cats = [cat_names[scores[i].argmax()] for i in range(len(scores))]
for cat, n in Counter(dominant_cats).most_common():
    print(f'  {cat:<22} {n} oggetti dominanti')

sims = scores

## 5. Assemblaggio ed export

In [132]:
frasi_out = []
for i, v in enumerate(resolved):
    frasi_out.append({
        'oggetto':         v['oggetto'],
        'testo':           v['testo'],
        'capitolo':        v['capitolo'],
        'titolo_capitolo': v['titolo_capitolo'],
        'categorie':       {cat: round(float(sims[i, j]), 4) for j, cat in enumerate(cat_names)},
    })

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
output = {
    'meta': {
        'titolo':       "L'isola del giorno prima",
        'autore':       'Umberto Eco',
        'gliner_model': GLINER_MODEL,
        'st_model':     ST_MODEL,
        'categorie':    cat_names,
        'n_capitoli':   len(real_chapters),
        'n_frasi':      len(frasi_out),
    },
    'frasi': frasi_out,
}

with open(OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f'Salvato: {OUT_PATH}')
print(f'Dimensione: {OUT_PATH.stat().st_size / 1024:.0f} KB')
print(f'Oggetti: {len(frasi_out):,}')

Salvato: ../app/static/isola.json
Dimensione: 139 KB
Oggetti: 222


## 6. Verifica

In [133]:
import math
from collections import Counter

# Distribuzione categoria dominante
dominant = Counter(max(f['categorie'], key=f['categorie'].get) for f in frasi_out)
print('=== Distribuzione categoria dominante ===')
for cat, n in dominant.most_common():
    pct = 100 * n / len(frasi_out)
    print(f'  {cat:<12} {n:4}  ({pct:5.1f}%)  {"█" * int(pct / 1.5)}')

# Entropia media (misura quanto le distribuzioni sono nette)
def entropy(d):
    return -sum(v * math.log(v + 1e-9) for v in d.values())

entropies = [entropy(f['categorie']) for f in frasi_out]
avg_h = sum(entropies) / len(entropies)
max_h = math.log(len(frasi_out[0]['categorie']))
print(f'\n=== Entropia media ===')
print(f'  {avg_h:.3f}  (max teorico: {max_h:.3f}  |  più bassa = assegnazioni più nette)')

# Oggetti con assegnazione debole (max score < 0.28)
WEAK_THRESHOLD = 0.28
weak = [(f, max(f['categorie'].values())) for f in frasi_out if max(f['categorie'].values()) < WEAK_THRESHOLD]
weak.sort(key=lambda x: x[1])
print(f'\n=== Assegnazioni deboli (max < {WEAK_THRESHOLD}) — {len(weak)} oggetti ===')
for f, score in weak[:25]:
    cat = max(f['categorie'], key=f['categorie'].get)
    print(f'  ({score:.3f})  {f["oggetto"]:<25}  → {cat}')

# Top 5 per ogni categoria
print('\n=== Top 5 per categoria ===')
cat_names_out = list(frasi_out[0]['categorie'].keys())
for cat in cat_names_out:
    ranked = sorted(frasi_out, key=lambda f: -f['categorie'][cat])[:5]
    print(f'\n  {cat}:')
    for f in ranked:
        dom = max(f['categorie'], key=f['categorie'].get)
        flag = '' if dom == cat else f'  ← dominante: {dom}'
        print(f'    ({f["categorie"][cat]:.3f})  {f["oggetto"]:<22}  {f["testo"][:60]}{flag}')

=== Distribuzione categoria dominante ===
  natura         65  ( 29.3%)  ███████████████████
  navigazione    44  ( 19.8%)  █████████████
  sguardo        30  ( 13.5%)  █████████
  amore          25  ( 11.3%)  ███████
  scrittura      20  (  9.0%)  ██████
  fede           19  (  8.6%)  █████
  calcolo        19  (  8.6%)  █████

=== Entropia media ===
  1.588  (max teorico: 1.946  |  più bassa = assegnazioni più nette)

=== Assegnazioni deboli (max < 0.28) — 0 oggetti ===

=== Top 5 per categoria ===

  calcolo:
    (0.891)  Orologio                Ma se non è difficile determinare l’ora del luogo del riliev
    (0.580)  Meridiano               E si chiama meridiano perché, ovunque un uomo stia e in qual
    (0.507)  Bussola                 Al tramonto, come un bighellone, passava dal timoniere, dove
    (0.492)  Seghe                   E con il legno da carpentiere mancavano gli arnesi di carpen
    (0.486)  Primo Meridiano         Qui.” “Ma perché qui?” “Perché qui è il meridiano cen